# Lab 03: Deploy CertAgent to Amazon Bedrock AgentCore Runtime

## Overview

In this lab, we deploy a persistent AI agent to **AgentCore Runtime** using the AgentCore CLI. Once deployed, the agent runs as a managed service that can be invoked on demand to perform certificate management operations.

### Architecture

```
User --> AgentCore Runtime (CertAgent) --> Lambda tools --> DynamoDB / Secrets Manager
```

The CertAgent is packaged with its tool definitions and deployed to AgentCore Runtime, which handles scaling, logging, and tracing. The agent invokes Lambda functions as tools to scan, renew, download, and store certificates.

**Estimated time:** 25 minutes

In [ ]:
import json, pathlib, boto3, os

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)
REPO_DIR = pathlib.Path(REPO_DIR)
print(f'Region: {AWS_REGION}')
print(f'Scan Lambda: {LAMBDA_SCAN}')
print('Config loaded')

## Step 1 -- Install Node.js and AgentCore CLI

AgentCore CLI is distributed as an npm package. We first ensure Node.js 20 is available, then install the CLI globally.

In [ ]:
%%bash
# Install Node.js 20 via conda (fastest in SageMaker)
conda install -y -c conda-forge nodejs=20 2>/dev/null || {
    echo "conda failed, trying nvm..."
    curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.1/install.sh | bash
    export NVM_DIR="$HOME/.nvm"
    [ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
    nvm install 20
}
node --version
npm --version

# Install AgentCore CLI
npm install -g @aws/agentcore
agentcore --version

## Step 2 -- Bootstrap AWS CDK

AgentCore deploys infrastructure using AWS CDK under the hood. We need to bootstrap CDK in the target account/region if it has not been done already. This creates the staging bucket and roles that CDK needs.

In [ ]:
%%bash
# Install CDK if not present
npm install -g aws-cdk 2>/dev/null
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)
REGION=$(aws configure get region || echo us-east-1)
echo "Bootstrapping CDK in $ACCOUNT / $REGION..."
cdk bootstrap aws://$ACCOUNT/$REGION 2>&1 | tail -5
echo "Done"

## Step 3 -- Review the CertAgent Project

The CertAgent project follows the AgentCore project layout:

- `agentcore/agentcore.json` -- Agent metadata and configuration
- `agentcore/aws-targets.json` -- Deployment targets (account + region)
- `app/CertAgent/main.py` -- Agent application code with tool definitions
- `app/CertAgent/pyproject.toml` -- Python dependencies

Let's inspect the structure and the main agent code.

In [ ]:
%%bash
echo "=== AgentCore Project Structure ==="
find ~/certagent-workshop/CertAgent -type f | sort | head -20
echo ""
echo "=== Agent Code (main.py first 40 lines) ==="
head -40 ~/certagent-workshop/CertAgent/app/CertAgent/main.py

## Step 4 -- Configure Deployment Target

We need to tell AgentCore which AWS account and region to deploy into. This is configured in `aws-targets.json`.

In [ ]:
import json

account_id = boto3.client('sts').get_caller_identity()['Account']
targets = [{"accountId": account_id, "region": AWS_REGION}]

targets_path = REPO_DIR / 'CertAgent' / 'agentcore' / 'aws-targets.json'
targets_path.write_text(json.dumps(targets, indent=2))
print(f'Targets set: account={account_id}, region={AWS_REGION}')

## Step 5 -- Set Lambda ARNs in Environment

The agent needs to know the ARNs of the Lambda functions it will invoke as tools. We write these into a `.env.local` file that AgentCore picks up during deployment.

In [ ]:
lm_client = boto3.client('lambda', region_name=AWS_REGION)

env_vars = {
    'LAMBDA_SCAN': LAMBDA_SCAN,
    'LAMBDA_RENEW': LAMBDA_RENEW,
    'LAMBDA_STATUS': LAMBDA_STATUS,
    'LAMBDA_DOWNLOAD': LAMBDA_DOWNLOAD,
    'LAMBDA_STORE': LAMBDA_STORE,
    'LAMBDA_INVENTORY': LAMBDA_INVENTORY,
    'AWS_REGION': AWS_REGION,
}

env_path = REPO_DIR / 'CertAgent' / 'agentcore' / '.env.local'
env_content = '\n'.join(f'{k}={v}' for k, v in env_vars.items())
env_path.write_text(env_content)
print('Written .env.local:')
print(env_content)

## Step 6 -- Deploy to AgentCore Runtime

Now we deploy the agent as a persistent service in AgentCore Runtime. This packages the agent code, provisions the necessary infrastructure (IAM roles, compute), and registers the agent so it can be invoked.

In [ ]:
%%bash
export PATH="$HOME/.nvm/versions/node/$(ls $HOME/.nvm/versions/node/ 2>/dev/null | head -1)/bin:$PATH:/opt/conda/bin"
cd ~/certagent-workshop/CertAgent
agentcore deploy -y 2>&1 | tail -30
echo ""
echo "=== Deployment complete ==="
agentcore status

## Step 7 -- Invoke the Deployed Agent

With the agent deployed, we can invoke it via the AgentCore CLI. Let's test several scenarios:

1. Scan for expiring certificates
2. Renew a specific certificate
3. Multi-step operation (scan then renew critical ones)

In [ ]:
%%bash
export PATH="$HOME/.nvm/versions/node/$(ls $HOME/.nvm/versions/node/ 2>/dev/null | head -1)/bin:$PATH:/opt/conda/bin"
cd ~/certagent-workshop/CertAgent
echo "--- Asking agent to scan certs ---"
agentcore invoke "What certificates are expiring soon? Use mock mode and give me a prioritized summary."

In [ ]:
%%bash
export PATH="$HOME/.nvm/versions/node/$(ls $HOME/.nvm/versions/node/ 2>/dev/null | head -1)/bin:$PATH:/opt/conda/bin"
cd ~/certagent-workshop/CertAgent
echo "--- Asking agent to renew ---"
agentcore invoke "Renew the certificate for api.example.com using mock mode"

In [ ]:
%%bash
export PATH="$HOME/.nvm/versions/node/$(ls $HOME/.nvm/versions/node/ 2>/dev/null | head -1)/bin:$PATH:/opt/conda/bin"
cd ~/certagent-workshop/CertAgent
echo "--- Multi-step: scan + renew critical ---"
agentcore invoke "Scan for certs expiring in 7 days with mock data, then renew any CRITICAL ones"

## Step 8 -- View Logs and Traces

AgentCore Runtime provides built-in observability. Let's check the recent logs and traces to see how the agent processed our requests.

In [ ]:
%%bash
export PATH="$HOME/.nvm/versions/node/$(ls $HOME/.nvm/versions/node/ 2>/dev/null | head -1)/bin:$PATH:/opt/conda/bin"
cd ~/certagent-workshop/CertAgent
echo "=== Recent logs ==="
agentcore logs --since 5m 2>&1 | head -30
echo ""
echo "=== Recent traces ==="
agentcore traces list 2>&1 | head -10

## Lab Complete

In this lab we:

1. **Installed** Node.js and the AgentCore CLI
2. **Bootstrapped** AWS CDK in our account
3. **Configured** the deployment target and Lambda ARNs
4. **Deployed** the CertAgent to AgentCore Runtime as a persistent service
5. **Invoked** the agent with scan, renew, and multi-step requests
6. **Viewed** logs and traces for observability

The agent is now running as a managed service and can be invoked programmatically or via the CLI at any time.

---

**Next:** `04_proactive_monitoring.ipynb` -- Set up proactive monitoring with EventBridge rules that trigger the agent automatically when certificates approach expiration.